# DAC Toy Model: Access Matrix, ACLs, and Delegation


## Teaching goal

This is a toy model of Discretionary Access Control (DAC). It shows how access can be represented
with an access matrix, then viewed as object-centered ACLs. Real operating systems, databases, and
healthcare applications already provide mature DAC or ACL mechanisms; system administrators normally
configure policy instead of implementing the enforcement engine themselves.


In [ ]:
from dataclasses import dataclass
from pprint import pprint


@dataclass(frozen=True)
class Request:
    user: str
    operation: str
    resource: str
    context: dict


users = {
    "dr_rossi": {"name": "Dr. Rossi", "department": "cardiology"},
    "nurse_amina": {"name": "Nurse Amina", "department": "ward-a"},
    "lab_tech": {"name": "Lab Technician", "department": "laboratory"},
    "billing_clerk": {"name": "Billing Clerk", "department": "billing"},
    "privacy_auditor": {"name": "Privacy Auditor", "department": "compliance"},
}

resources = {
    "ehr_note_42": {"type": "ehr_note", "patient": "patient-42", "department": "cardiology"},
    "lab_result_42": {"type": "lab_result", "patient": "patient-42", "department": "laboratory"},
    "billing_record_42": {"type": "billing_record", "patient": "patient-42", "department": "billing"},
    "audit_log": {"type": "audit_log", "patient": None, "department": "compliance"},
}

requests = [
    Request("dr_rossi", "read", "ehr_note_42", {"assigned_patient": True, "emergency": False}),
    Request("nurse_amina", "write", "ehr_note_42", {"assigned_patient": True, "emergency": False}),
    Request("lab_tech", "write", "lab_result_42", {"assigned_patient": False, "emergency": False}),
    Request("billing_clerk", "read", "ehr_note_42", {"assigned_patient": False, "emergency": False}),
    Request("privacy_auditor", "read", "audit_log", {"assigned_patient": False, "emergency": False}),
]


In [ ]:
# The access matrix maps each user to each resource and the operations allowed there.
access_matrix = {
    "dr_rossi": {
        "ehr_note_42": {"read", "write"},
        "lab_result_42": {"read"},
    },
    "nurse_amina": {
        "ehr_note_42": {"read", "write"},
        "lab_result_42": {"read"},
    },
    "lab_tech": {
        "lab_result_42": {"read", "write"},
    },
    "billing_clerk": {
        "billing_record_42": {"read", "write"},
    },
    "privacy_auditor": {
        "audit_log": {"read"},
    },
}


def dac_allows(request: Request) -> bool:
    # DAC checks the requesting identity and the rights assigned to that identity.
    user_rights = access_matrix.get(request.user, {})
    resource_rights = user_rights.get(request.resource, set())
    return request.operation in resource_rights


for request in requests:
    print(request.user, request.operation, request.resource, "=>", dac_allows(request))


In [ ]:
# The same matrix can be viewed as ACLs: each resource lists who may do what.
def build_acls(matrix):
    acls = {}
    for user, resource_map in matrix.items():
        for resource, operations in resource_map.items():
            acls.setdefault(resource, {})[user] = sorted(operations)
    return acls


acls = build_acls(access_matrix)
pprint(acls)


In [ ]:
# DAC can allow delegation: someone with enough control may share access.
def delegate(owner: str, grantee: str, resource: str, operations: set[str]) -> None:
    # In a real system this delegation would itself be authorized and audited.
    if "write" not in access_matrix.get(owner, {}).get(resource, set()):
        raise PermissionError(f"{owner} cannot delegate access to {resource}")
    access_matrix.setdefault(grantee, {}).setdefault(resource, set()).update(operations)


print("Before delegation:", dac_allows(Request("billing_clerk", "read", "ehr_note_42", {})))
delegate("dr_rossi", "billing_clerk", "ehr_note_42", {"read"})
print("After delegation:", dac_allows(Request("billing_clerk", "read", "ehr_note_42", {})))


## What to notice

DAC is flexible, especially for collaboration. The risk is that access can spread if delegation is
not controlled. In production, administrators define sharing rules, ownership rules, and audit
expectations; they do not hand-code an access matrix inside an application notebook.
